# BNPL Project — Analysis 1
## Weighted Descriptive Comparison


---
## Setup — Imports and dataset load

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.multitest import multipletests

# Display options
pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 140)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

print('Libraries loaded.')

Libraries loaded.


In [2]:
import zipfile
print("Loading data...")
z = zipfile.ZipFile('../data/raw/public2024.csv.zip')
df = pd.read_csv(z.open('public2024.csv'), low_memory=False)
print(f'Loaded: {df.shape[0]:,} rows  x  {df.shape[1]:,} columns')

Loading data...


Loaded: 12,295 rows  x  751 columns


---
# Analysis 1 — Weighted descriptive comparison + chi-square tests

**Purpose.** Exploratory data analysis to establish whether BNPL use varies across demographic and financial-vulnerability groups.

### Step 1.1 — Sanity checks

Three things to verify:
1. Sample size matches methodology (12,295).
2. `BNPL1` has ~1,670 "Yes" responses.
3. Survey weight is normalized to mean ≈ 1.

In [3]:
print('--- BNPL1 (used BNPL in past year) ---')
print(df['BNPL1'].value_counts(dropna=False))
print(f'Missing: {df["BNPL1"].isna().sum()}')

--- BNPL1 (used BNPL in past year) ---
BNPL1
No     10625
Yes     1670
Name: count, dtype: int64
Missing: 0


In [4]:
print('--- weight ---')
print(df['weight'].describe())
print(f'\nSum of weights: {df["weight"].sum():,.1f}  (should be ≈ N = {len(df):,})')

--- weight ---
count   12295.0000
mean        1.0000
std         0.4351
min         0.1788
25%         0.7163
50%         0.9244
75%         1.1727
max         3.5159
Name: weight, dtype: float64

Sum of weights: 12,295.0  (should be ≈ N = 12,295)


### Step 1.2 — Build the analysis dataframe

Two recodes:
- `BNPL1`: "Yes"/"No" → 1/0 numeric.
- Predictors: "Refused" / -1 codes → `NaN`.

In [5]:
predictors_demo = ['ppagecat', 'ppgender', 'race_5cat', 'educ_4cat',
                   'ppinc7', 'ppmarit5', 'ppmsacat']
predictors_vuln = ['B0_b', 'B2', 'EF1', 'EF3_f', 'EF7']

d = df.copy()

# Outcome: Yes/No -> 1/0
d['BNPL1_bin'] = d['BNPL1'].map({'Yes': 1, 'No': 0})

# Treat SHED's refusal codes as missing on predictors
REFUSED_TOKENS = {'Refused', -1, '-1'}
for col in predictors_demo + predictors_vuln:
    d[col] = d[col].where(~d[col].isin(REFUSED_TOKENS), np.nan)

print(d['BNPL1_bin'].value_counts(dropna=False))
overall_rate = np.average(d['BNPL1_bin'], weights=d['weight'])
print(f'\nWeighted BNPL rate (overall): {overall_rate:.3%}')

BNPL1_bin
0    10625
1     1670
Name: count, dtype: int64

Weighted BNPL rate (overall): 14.683%


### Step 1.3 — Weighted bivariate tables

For each predictor X, weighted proportion of BNPL users by levels of X.

In [6]:
def weighted_bnpl_by(df_, predictor, outcome='BNPL1_bin', weight='weight'):
    """Weighted BNPL rate by levels of `predictor`."""
    sub = df_[[predictor, outcome, weight]].dropna()
    rows = []
    for level, g in sub.groupby(predictor, observed=True):
        w = g[weight].values
        y = g[outcome].values
        rows.append({
            'predictor': predictor,
            'level': level,
            'n_raw': len(g),
            'n_weighted': w.sum(),
            'bnpl_rate_weighted': np.average(y, weights=w),
        })
    out = pd.DataFrame(rows).sort_values('bnpl_rate_weighted', ascending=False)
    return out.reset_index(drop=True)

weighted_bnpl_by(d, 'ppagecat')

,predictor,level,n_raw,n_weighted,bnpl_rate_weighted
0,ppagecat,25-34,1799,2048.1917,0.1996
1,ppagecat,35-44,2015,2328.5607,0.1836
2,ppagecat,18-24,886,1252.7699,0.1789
3,ppagecat,45-54,1774,1775.5418,0.1606
4,ppagecat,55-64,2302,2101.7118,0.1283
5,ppagecat,65-74,2219,1776.6420,0.0835
6,ppagecat,75+,1300,1011.5533,0.0411


In [7]:
all_predictors = predictors_demo + predictors_vuln
bivariate_tables = {p: weighted_bnpl_by(d, p) for p in all_predictors}

for p, tbl in bivariate_tables.items():
    print(f'\n=== BNPL rate by {p} ===')
    print(tbl.to_string(index=False))


=== BNPL rate by ppagecat ===
predictor level  n_raw  n_weighted  bnpl_rate_weighted
 ppagecat 25-34   1799   2048.1917              0.1996
 ppagecat 35-44   2015   2328.5607              0.1836
 ppagecat 18-24    886   1252.7699              0.1789
 ppagecat 45-54   1774   1775.5418              0.1606
 ppagecat 55-64   2302   2101.7118              0.1283
 ppagecat 65-74   2219   1776.6420              0.0835
 ppagecat   75+   1300   1011.5533              0.0411

=== BNPL rate by ppgender ===
predictor  level  n_raw  n_weighted  bnpl_rate_weighted
 ppgender Female   6065   6298.2504              0.1696
 ppgender   Male   6230   5996.7208              0.1229

=== BNPL rate by race_5cat ===
predictor    level  n_raw  n_weighted  bnpl_rate_weighted
race_5cat    Black   1377   1488.4256              0.2515
race_5cat Hispanic   1685   2201.5319              0.2056
race_5cat    Other    524    363.0922              0.1386
race_5cat    Asian    528    756.1047              0.1185
race_5ca

### Step 1.4 — Rao-Scott corrected chi-square

Standard Pearson chi-square assumes simple random sampling. SHED uses post-stratification weights, so we apply the **first-order Rao-Scott correction** via the standard rescaled-table approximation: build a weighted contingency table, rescale so its grand total equals the unweighted N, then call `scipy.stats.chi2_contingency`. This adjusts the test statistic for the design effect of weighting.

(The full Rao-Scott requires PSU/strata variables, which SHED's public file does not release; rescaling is the conventional fallback when only post-stratification weights are published.)

In [8]:
def weighted_chi2(df_, predictor, outcome='BNPL1_bin', weight='weight'):
    """Design-corrected chi-square via rescaled weighted contingency table."""
    sub = df_[[predictor, outcome, weight]].dropna()
    ctab = (sub
            .groupby([predictor, outcome], observed=True)[weight]
            .sum()
            .unstack(fill_value=0))
    n_unweighted = len(sub)
    ctab_scaled = ctab * (n_unweighted / ctab.values.sum())
    chi2, p, dof, _ = stats.chi2_contingency(ctab_scaled)
    return {'predictor': predictor, 'n': n_unweighted,
            'chi2': chi2, 'dof': dof, 'p_value': p}

results = [weighted_chi2(d, p) for p in all_predictors]
results_df = pd.DataFrame(results)
results_df['domain'] = results_df['predictor'].apply(
    lambda p: 'demographic' if p in predictors_demo else 'vulnerability')
results_df = results_df[['domain', 'predictor', 'n', 'dof', 'chi2', 'p_value']]
results_df.sort_values(['domain', 'p_value'])

,domain,predictor,n,dof,chi2,p_value
2,demographic,race_5cat,12295,4,268.2194,0.0000
0,demographic,ppagecat,12295,6,236.6415,0.0000
5,demographic,ppmarit5,12295,4,128.4166,0.0000
4,demographic,ppinc7,12295,6,119.7707,0.0000
3,demographic,educ_4cat,12295,3,95.7671,0.0000
1,demographic,ppgender,12295,1,53.0564,0.0000
6,demographic,ppmsacat,12295,1,0.0433,0.8351
11,vulnerability,EF7,12295,4,564.3385,0.0000
8,vulnerability,B2,12295,3,413.1505,0.0000
9,vulnerability,EF1,12295,1,352.7483,0.0000


### Step 1.5 — Holm-Bonferroni correction

Applied within each family (demographic, vulnerability) separately, per methodology.

In [9]:
corrected = []
for domain, grp in results_df.groupby('domain'):
    reject, p_adj, _, _ = multipletests(grp['p_value'].values,
                                        alpha=0.05, method='holm')
    out = grp.copy()
    out['p_value_holm'] = p_adj
    out['sig_holm_0.05'] = reject
    corrected.append(out)

results_final = pd.concat(corrected).sort_values(['domain', 'p_value'])
results_final

,domain,predictor,n,dof,chi2,p_value,p_value_holm,sig_holm_0.05
2,demographic,race_5cat,12295,4,268.2194,0.0000,0.0000,True
0,demographic,ppagecat,12295,6,236.6415,0.0000,0.0000,True
5,demographic,ppmarit5,12295,4,128.4166,0.0000,0.0000,True
4,demographic,ppinc7,12295,6,119.7707,0.0000,0.0000,True
3,demographic,educ_4cat,12295,3,95.7671,0.0000,0.0000,True
1,demographic,ppgender,12295,1,53.0564,0.0000,0.0000,True
6,demographic,ppmsacat,12295,1,0.0433,0.8351,0.8351,False
11,vulnerability,EF7,12295,4,564.3385,0.0000,0.0000,True
8,vulnerability,B2,12295,3,413.1505,0.0000,0.0000,True
9,vulnerability,EF1,12295,1,352.7483,0.0000,0.0000,True


In [10]:
# Clean table for the writeup
report_tbl = results_final[['domain', 'predictor', 'n', 'dof',
                            'chi2', 'p_value', 'p_value_holm',
                            'sig_holm_0.05']].copy()
report_tbl.columns = ['Domain', 'Predictor', 'N', 'df',
                      'Chi-square', 'p (raw)', 'p (Holm)',
                      'Sig. at .05']
report_tbl.reset_index(drop=True, inplace=True)
report_tbl

,Domain,Predictor,N,df,Chi-square,p (raw),p (Holm),Sig. at .05
0,demographic,race_5cat,12295,4,268.2194,0.0000,0.0000,True
1,demographic,ppagecat,12295,6,236.6415,0.0000,0.0000,True
2,demographic,ppmarit5,12295,4,128.4166,0.0000,0.0000,True
3,demographic,ppinc7,12295,6,119.7707,0.0000,0.0000,True
4,demographic,educ_4cat,12295,3,95.7671,0.0000,0.0000,True
5,demographic,ppgender,12295,1,53.0564,0.0000,0.0000,True
6,demographic,ppmsacat,12295,1,0.0433,0.8351,0.8351,False
7,vulnerability,EF7,12295,4,564.3385,0.0000,0.0000,True
8,vulnerability,B2,12295,3,413.1505,0.0000,0.0000,True
9,vulnerability,EF1,12295,1,352.7483,0.0000,0.0000,True


In [11]:
# Analysis 1 outputs are kept in memory only:
#   - report_tbl              : the chi-square summary table
#   - bivariate_tables[<pred>]: BNPL rate by each predictor
# Both can be referenced directly in later cells or in the report.
print('Analysis 1 results held in memory:')
print(f'  report_tbl              -> chi-square table ({len(report_tbl)} rows)')
print(f'  bivariate_tables (dict) -> rate tables for {len(bivariate_tables)} predictors:')
for p in bivariate_tables:
    print(f'      bivariate_tables["{p}"]')

Analysis 1 results held in memory:
  report_tbl              -> chi-square table (12 rows)
  bivariate_tables (dict) -> rate tables for 12 predictors:
      bivariate_tables["ppagecat"]
      bivariate_tables["ppgender"]
      bivariate_tables["race_5cat"]
      bivariate_tables["educ_4cat"]
      bivariate_tables["ppinc7"]
      bivariate_tables["ppmarit5"]
      bivariate_tables["ppmsacat"]
      bivariate_tables["B0_b"]
      bivariate_tables["B2"]
      bivariate_tables["EF1"]
      bivariate_tables["EF3_f"]
      bivariate_tables["EF7"]
